In [0]:
import pyspark.sql.functions as F

SCHEMA = "workspace.erp_lakehouse"

In [0]:
orders = spark.table(f"{SCHEMA}.silver_b_orders")
order_lines = spark.table(f"{SCHEMA}.silver_b_order_lines")

#join the two tables silver_b_orders and silver_b_order_lines
joined = orders.join(order_lines, on="order_id", how="inner")
#display(joined)


joined = joined.withColumn("line_total", F.col("qty") * F.col("unit_price"))
#display(joined)
daily = (joined
         .groupBy("order_date")
         .agg(
             F.countDistinct("order_id").alias("order_count"),
             F.count("*").alias("line_count"),
             F.sum("qty").alias("total_units"),
             F.sum("line_total").alias("total_sales")
         )     
         )
daily = daily.orderBy(F.col("order_date").desc())

display(daily)


daily.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{SCHEMA}.gold_sales_by_day")


![image_1794443276995.png](../docs/images/image_1794443276995.png)